# ہوٹل بکنگ بذریعہ پرائیورٹی ممبر مڈل ویئر

یہ نوٹ بک Microsoft Agent Framework استعمال کرتے ہوئے **فنکشن پر مبنی مڈل ویئر** کا مظاہرہ کرتا ہے۔ ہم مشروط ورک فلو کی مثال پر توجہ مرکوز کرتے ہوئے ایک مڈل ویئر لیئر شامل کرتے ہیں جو پرائیورٹی ممبرز کو خاص مراعات دیتی ہے۔

## آپ کیا سیکھیں گے:
1. **فنکشن پر مبنی مڈل ویئر**: فنکشن کے نتائج کو روکنا اور ترمیم کرنا
2. **کانٹیکسٹ تک رسائی**: عمل کے بعد `context.result` کو پڑھنا اور ترمیم کرنا
3. **کاروباری منطق کا نفاذ**: پرائیورٹی ممبر کے فائدے
4. **نتیجے کا اوور رائیڈ**: صارف کی حیثیت کی بنیاد پر فنکشن کے نتائج میں تبدیلی
5. **ایک ہی ورک فلو، مختلف نتائج**: مڈل ویئر کی بنیاد پر رویہ میں تبدیلیاں

## مڈل ویئر کے ساتھ ورک فلو فن تعمیر:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## مشروط ورک فلو سے کلیدی فرق:

**مڈل ویئر کے بغیر** (14-conditional-workflow.ipynb):
- پیرس میں کوئی کمرے نہیں → alternative_agent کی طرف راستہ

**مڈل ویئر کے ساتھ** (یہ نوٹ بک):
- عمومی صارف + پیرس → کوئی کمرے نہیں → alternative_agent کی طرف راستہ
- پرائیورٹی صارف + پیرس → 🌟 مڈل ویئر اوور رائیڈ! → دستیاب → booking_agent کی طرف راستہ

## پیشگی ضروریات:
- Microsoft Agent Framework نصب
- مشروط ورک فلو کی سمجھ (دیکھیں 14-conditional-workflow.ipynb)
- GitHub ٹوکن یا OpenAI API کلید
- مڈل ویئر پیٹرنز کی بنیادی سمجھ


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## مرحلہ 1: ساختہ نتائج کے لیے Pydantic ماڈلز کی تعریف کریں

یہ ماڈلز وہ **اسکیمہ** تعریف کرتے ہیں جو ایجنٹس واپس کریں گے۔ ہم نے `priority_override` فیلڈ شامل کیا ہے تاکہ یہ پتہ چلایا جا سکے کہ کب مڈل ویئر نے دستیابی کے نتیجے میں تبدیلی کی ہے۔


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## مرحلہ 2: پریارٹی ممبرز ڈیٹابیس کی تعریف کریں

اس ڈیمو کے لیے، ہم پریارٹی ممبرشپ ڈیٹابیس کی نقل کرنے جا رہے ہیں۔ پروڈکشن میں، یہ ایک حقیقی ڈیٹابیس یا API کو استفسار کرے گا۔

**پریارٹی ممبرز:**
- `alice@example.com` - وی آئی پی ممبر
- `bob@example.com` - پریمیم ممبر  
- `priority_user` - ٹیسٹ اکاؤنٹ


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## مرحلہ 3: ہوٹل بکنگ ٹول بنائیں

وہی جیسا کہ مشروط ورک فلو، لیکن اب یہ مڈل ویئر کے ذریعے روکا جائے گا!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Step 4: 🌟 پریوریٹی چیک مڈل ویئر بنائیں (اہم خصوصیت!)

یہ اس نوٹ بک کی **بنیادی فعالیت** ہے۔ مڈل ویئر:

1. فنکشن hotel_booking کی کال کو **روکتا ہے**
2. `next(context)` کو کال کرکے فنکشن کو معمول کے مطابق **چلاتا ہے**
3. `context.result` میں نتیجہ کا **معائنہ** کرتا ہے
4. اگر صارف کو ترجیح دی گئی ہو اور کوئی کمرا دستیاب نہ ہو تو نتیجہ کو **اووررائیڈ** کرتا ہے
5. تبدل شدہ نتیجہ کو ایجنٹ کو **واپس بھیجتا ہے**

**اہم پیٹرن:**
```python
async def my_middleware(context, next):
    await next(context)  # فنکشن کو چلائیں
    # اب context.result میں فنکشن کا نتیجہ شامل ہے
    if some_condition:
        context.result = new_value  # اوور رائیڈ کریں!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## Step 5: راستے کے لیے کنڈیشن فنکشنز کی تعریف کریں

ویسا ہی کنڈیشن فنکشنز جیسے کہ conditional workflow میں ہوتے ہیں - یہ ساخت شدہ آؤٹ پٹ کا معائنہ کرتے ہیں تاکہ راستہ طے کیا جا سکے۔


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## Step 6: کسٹم ڈسپلے ایگزیکیوٹر بنائیں

پہلے جیسے ہی ایگزیکیوٹر - حتمی ورک فلو آؤٹ پٹ دکھاتا ہے۔


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## مرحلہ 7: ماحول کی متغیرات لوڈ کریں

LLM کلائنٹ کو تشکیل دیں (GitHub ماڈلز یا OpenAI)۔


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## مرحلہ 8: مڈل ویئر کے ساتھ AI ایجنٹس بنائیں

**اہم فرق:** جب availability_agent بناتے ہیں، تو ہم `middleware` پیرامیٹر دیتے ہیں!

یہی طریقہ ہے جس سے ہم priority_check_middleware کو ایجنٹ کے فنکشن انووکیشن پائپ لائن میں داخل کرتے ہیں۔


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## مرحلہ 9: ورک فلو تعمیر کریں

پہلے کی طرح ہی ورک فلو کا ڈھانچہ - دستیابی کی بنیاد پر مشروط راستہ۔


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## Step 10: Test Case 1 - پیرس میں معمول کا صارف (کوئی اوور رائیڈ نہیں)

ایک معمول کا صارف پیرس بک کرنے کی کوشش کرتا ہے → کمروں کی کوئی دستیابی نہیں → alternative_agent کو روٹنگ ہوتی ہے


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## Step 11: Test Case 2 - 🌟 پیرس میں ترجیحی صارف (اوور رائیڈ کے ساتھ!)

ایک ترجیحی رکن پیرس کی بکنگ کرنے کی کوشش کرتا ہے → ابتدا میں کوئی کمرہ دستیاب نہیں → 🌟 مڈل ویئر اوور رائیڈ کر دیتا ہے! → booking_agent کی طرف راہنمائی

**یہ مڈل ویئر کی طاقت کا اہم مظاہرہ ہے!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## مرحلہ 12: ٹیسٹ کیس 3 - اسٹاک ہوم میں پرائرٹی یوزر (پہلے سے دستیاب)

پرائرٹی یوزر اسٹاک ہوم کی کوشش کرتا ہے → کمرے دستیاب ہیں → اووررائیڈ کی ضرورت نہیں → بُکنگ_ایجنٹ کی طرف راستہ دیتا ہے

یہ ظاہر کرتا ہے کہ مڈل ویئر صرف تب عمل کرتا ہے جب ضرورت ہو!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## اہم نکات اور مڈل ویئر کے تصورات

### ✅ آپ نے کیا سیکھا:

#### **1. فنکشن پر مبنی مڈل ویئر پیٹرن**

مڈل ویئر فنکشن کالز کو ایک سادہ async فنکشن کے ذریعے انٹرسیپٹ کرتا ہے:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # فنکشن کے عملدرآمد سے پہلے
    print("Intercepting...")
    
    # فنکشن کو چلائیں
    await next(context)
    
    # فنکشن کے عملدرآمد کے بعد - نتیجہ کا معائنہ کریں
    if context.result:
        # اگر ضرورت ہو تو نتیجہ تبدیل کریں
        context.result = modified_value
```

#### **2. کانٹیکسٹ تک رسائی اور نتیجہ اوور رائیڈ کرنا**

- `context.function` - کال کیے جانے والے فنکشن تک رسائی
- `context.arguments` - فنکشن دلائل پڑھنا
- `context.kwargs` - اضافی پیرامیٹرز تک رسائی
- `await next(context)` - فنکشن کو چلائیں
- `context.result` - فنکشن کے آؤٹ پٹ کو پڑھنا/ترمیم کرنا

#### **3. بزنس لاجک کا نفاذ**

ہمارا مڈل ویئر ترجیحی ممبر کے فوائد کو نافذ کرتا ہے:
- **معمول کے صارفین**: کوئی تبدیلی نہیں، معیاری ورک فلو
- **ترجیحی صارفین**: "کوئی دستیابی نہیں" کو "دستیاب" سے اوور رائیڈ کرنا
- **مشروط لاجک**: صرف جب ضرورت ہو اوور رائیڈ کرتا ہے

#### **4. ایک ہی ورک فلو، مختلف نتائج**

مڈل ویئر کی طاقت:
- ✅ ورک فلو کی ساخت میں کوئی تبدیلی نہیں
- ✅ ٹول فنکشن میں کوئی تبدیلی نہیں
- ✅ مشروط روٹنگ لاجک میں کوئی تبدیلی نہیں
- ✅ صرف مڈل ویئر → مختلف رویہ!

### 🚀 حقیقی دنیا کی درخواستیں:

1. **وی آئی پی/پریمیئم خصوصیات**
   - پریمیئم صارفین کے لیے ریٹ لمٹس کو اوور رائیڈ کریں
   - وسائل تک ترجیحی رسائی فراہم کریں
   - ڈائنامک طور پر پریمیئم خصوصیات کھولیں

2. **A/B ٹیسٹنگ**
   - صارفین کو مختلف امپلیمینٹیشنز کی طرف روٹ کریں
   - مخصوص صارفین کے ساتھ نئی خصوصیات آزمائیں
   - خصوصیات کی تدریجی تعارف

3. **سیکیورٹی اور مطابقت**
   - فنکشن کالز کا آڈٹ کریں
   - حساس آپریشنز کو بلاک کریں
   - کاروباری قواعد نافذ کریں

4. **کارکردگی کی اصلاح**
   - مخصوص صارفین کے لیے نتائج کو کیش کریں
   - جہاں ممکن ہو مہنگے آپریشنز کو چھوڑیں
   - وسائل کی ڈائنامک الاٹمنٹ

5. **ایرر ہینڈلنگ اور ریٹری**
   - غلطیوں کو قید کریں اور مہذب طریقے سے سنبھالیں
   - ریٹری لاجک نافذ کریں
   - متبادل امپلیمینٹیشنز پر فال بیک کریں

6. **لاگنگ اور مانیٹرنگ**
   - فنکشن کی اجرا کے اوقات کو ٹریک کریں
   - پیرامیٹرز اور نتائج لاگ کریں
   - استعمال کے پیٹرنز کی نگرانی کریں

### 🔑 ڈیکوریٹرز سے اہم فرق:

| خصوصیت | ڈیکوریٹر | مڈل ویئر |
|---------|-----------|------------|
| **دائرہ کار** | واحد فنکشن | ایجنٹ کے تمام فنکشنز |
| **لچک** | تعین پر مقرر | رن ٹائم پر ڈائنامک |
| **کانٹیکسٹ** | محدود | مکمل ایجنٹ کا کانٹیکسٹ |
| **ترکیب** | متعدد ڈیکوریٹرز | مڈل ویئر پائپ لائن |
| **ایجنٹ سے واقفیت** | نہیں | ہاں (ایجنٹ کی حالت تک رسائی) |

### 📚 مڈل ویئر کب استعمال کریں:

✅ **مڈل ویئر استعمال کریں جب:**
- آپ کو صارف/سیشن کی حالت کی بنیاد پر رویہ میں ترمیم کرنی ہو
- آپ نے متعدد فنکشنز پر لاجک لاگو کرنی ہو
- ایجنٹ کی سطح کے کانٹیکسٹ تک رسائی چاہیے ہو
- آپ کراس کٹنگ خدشات (لاگنگ، توثیق وغیرہ) نافذ کر رہے ہوں

❌ **مڈل ویئر استعمال نہ کریں جب:**
- سادہ ان پٹ ویلیڈیشن ہو (Pydantic استعمال کریں)
- فنکشن کی مخصوص لاجک ہو (فنکشن میں ہی رکھیں)
- ایک بار کی ترامیم ہوں (بس فنکشن کو تبدیل کریں)

### 🎓 جدید پیٹرنز:

```python
# متعدد مڈل ویئر (عمل درآمد کا حکم اہم ہے!)
middleware=[
    logging_middleware,      # سب سے پہلے لاگز
    auth_middleware,         # پھر توثیق چیک کرتا ہے
    cache_middleware,        # پھر کیش چیک کرتا ہے
    rate_limit_middleware,   # پھر ریٹ لمٹس
    priority_check_middleware  # آخر میں ترجیح کی جانچ
]

# مشروط مڈل ویئر کا عمل
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # نتیجہ میں ترمیم کریں
    else:
        # عمل درآمد کو مکمل طور پر چھوڑ دیں
        context.result = cached_value
```

### 🔗 متعلقہ تصورات:

- **ایجنٹ مڈل ویئر**: agent.run() کالز کو انٹرسیپٹ کرتا ہے
- **فنکشن مڈل ویئر**: ٹول فنکشن کالز کو انٹرسیپٹ کرتا ہے (جو ہم نے استعمال کیا!)
- **مڈل ویئر پائپ لائن**: مڈل ویئر کی ایک چین جو ترتیب سے چلتی ہے
- **کانٹیکسٹ پرپیگیشن**: مڈل ویئر چین میں حالت کو منتقل کرنا


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
